# supervised variant prediction

In [1]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy.stats
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from joblib import Parallel, delayed
import os, glob, joblib, gc, esm , torch, ast, numpy as np, pandas as pd


In [2]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV,RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform

## select protein-coding gene

In [3]:
gene_name='tlyA'
# pick the reference CDS protein sequence
ref_seqs=pd.read_csv('./data/catalog/protein_sequences.csv')
ref_protein = ref_seqs.loc[ref_seqs['gene'] == gene_name, 'protein_sequence'].values

In [4]:
#load the gene sequence file
gene_sequences=pd.read_csv(f"./data/sequence_data_csv/{gene_name}_combined_sequence_data.csv")
gene_sequences_filtered = gene_sequences[gene_sequences["Frameshift_Mutation"] == 0].copy()
# Ensure required columns exist
if "Protein_Sequence" not in gene_sequences_filtered.columns:
    raise ValueError("CSV file must contain a 'Protein_Sequence' column.")

## train classical regressor models frome esm embedding : KNN, SVR, Random Forest

In [5]:
# Directory to save results
RESULTS_DIR = "gene_results"
os.makedirs(RESULTS_DIR, exist_ok=True)


#  Step 1: Convert String Embeddings to Numerical Arrays
def parse_tensor(tensor_str):
    """Convert string tensor representation into a NumPy array."""
    try:
        return np.array(ast.literal_eval(tensor_str.replace("tensor(", "").replace(")", "")))
    except Exception as e:
        print(f"Error parsing tensor: {tensor_str} -> {e}")
        return np.zeros(320)  # Adjust size if needed


#  Step 2: Load and Merge Data
def load_embeddings(df, column_name):
    """
    Load embeddings from a DataFrame and convert to numerical arrays.
    """
    # df["mean_embedding"] = df["mean_embedding"].apply(parse_tensor)
    df[column_name] = df[column_name].apply(parse_tensor)
    return df



#  Step 3: Extract Features (X) and Target (y)
def prepare_data(df, embedding_column="pooled_embedding", label_column="Phenotype"):
    """
    Prepare feature matrix (X) and target labels (y) for training.
    Filters for binary classification: only 'R' and 'S' phenotypes.
    """
    # Filter for binary classes in phenotype
    df_binary = df[df[label_column].isin(["R", "S"])].copy()

    # Encode labels (R -> 1, S -> 0)
    y = df_binary[label_column].astype("category").cat.codes

    # Stack embedding vectors
    X = np.vstack(df_binary[embedding_column].values)

    print(f"Binary classification only. Data prepared. Shape X: {X.shape}, y: {y.shape}, Classes: {np.unique(y)}")
    return X, y



#  Step 4: Train-Test Split
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)


#  Step 5: PCA Visualization and Save

def visualize_pca(X_train, y_train, gene_name, num_components=60):
    """
    Perform PCA and save the visualization.
    """
    
    if gene_name=='gyrB':
        n_train = X_train.shape[0] 
        n_components=randint(5, min(n_train, X_train.shape[1]))
        pca = PCA(n_components=n_components)
    else:
        pca = PCA(n_components=num_components)                    
    X_train_pca = pca.fit_transform(X_train)

    plt.figure(figsize=(7, 6))
    sc = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, marker='.')
    plt.xlabel("PCA First Principal Component")
    plt.ylabel("PCA Second Principal Component")
    plt.colorbar(sc, label="Variant Effect")
    plt.title(f"PCA Visualization for {gene_name}")

    # Save PCA Figure
    pca_path = os.path.join(RESULTS_DIR, f"{gene_name}_pca.png")
    plt.savefig(pca_path)
    plt.close()
    
    return pca



# ------------------------------------------------------------
#  RandomizedSearchCV instead of GridSearchCV  (ASCII only!)
# ------------------------------------------------------------

def run_random_search(
        X_train, y_train,
        num_pca_components: int = 60,
        n_iter: int = 25,
        cv_folds: int = 3,
        random_state: int = 42
):
    n_samples, n_features = X_train.shape
    max_components = min(num_pca_components, n_samples)

    pipe = Pipeline([
        ('pca', PCA(n_components=max_components)),
        ('model', 'passthrough')
    ])
    """
    Faster hyper‑parameter tuning with RandomizedSearchCV.
    """

    # ---------- search spaces (all ASCII hyphens) -----------------
    knn_dist = {
        'model': [KNeighborsRegressor()],
        'model__n_neighbors': randint(3, 31),
        'model__weights': ['uniform', 'distance'],
        'model__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'model__leaf_size': randint(10, 50),
        'model__p': randint(1, 3),
        'pca__n_components': randint(5, min(X_train.shape[0] * (cv_folds - 1) // cv_folds, X_train.shape[1]))
    }

    svr_dist = {
            'model': [SVR()],
            'model__C': [0.1, 1.0],
            'model__kernel': ['linear', 'rbf'],
            'model__degree': [3],
            'model__gamma': ['scale'],
            'pca__n_components': randint(5, min(X_train.shape[0] * (cv_folds - 1) // cv_folds, X_train.shape[1]))
        }
    rf_dist = {
        'model': [RandomForestRegressor()],
        'model__n_estimators': randint(50, 301),          # 50–300
        'model__max_features': ['sqrt', 'log2'],
        'model__min_samples_split': randint(2, 11),       # 2–10
        'model__min_samples_leaf': randint(1, 6),         # 1–5
        'model__criterion': ['squared_error'],
        'pca__n_components': randint(5, min(X_train.shape[0] * (cv_folds - 1) // cv_folds, X_train.shape[1]))
    }

    search_spaces = [knn_dist, svr_dist, rf_dist]
    models        = [KNeighborsRegressor, SVR, RandomForestRegressor]


    results, searches = [], []

    for cls, param_dist in zip(models, search_spaces):
        print(f"Running Randomized Search for {cls.__name__} …")

        rs = RandomizedSearchCV(
            estimator           = pipe,
            param_distributions = param_dist,
            n_iter              = n_iter,
            cv                  = cv_folds,
            random_state        = random_state,
            n_jobs              = -1,
            verbose             = 1,
            scoring             = 'r2'
        )
        rs.fit(X_train, y_train)
        results.append(pd.DataFrame(rs.cv_results_))
        searches.append(rs)

    return results, searches



#  Step 8: Evaluate Best Models using AUC and Save Results
def evaluate_models(grid_list, X_test, y_test, gene_name):
    """
    Evaluate the best models using AUC and save results.
    """
    results = []

    for grid in grid_list:
        best_model = grid.best_estimator_.get_params()["steps"][1][1]
        preds = grid.predict(X_test)

        try:
            auc = roc_auc_score(y_test, preds)
        except ValueError as e:
            print(f"[Warning] AUC could not be computed for {gene_name}: {e}")
            auc = None

        results.append({
            "Gene": gene_name,
            "Best Model": str(best_model),
            "AUC": auc
        })

        print(f"Best Model for {gene_name}: {best_model}")
        if auc is not None:
            print(f"AUC: {auc:.4f}")
        else:
            print("AUC: N/A (only one class present in test set)")
        print("\n", "-" * 80, "\n")

    return results



In [6]:
# Step 8: Run for Each Gene and Save All Results
def main(gene_df):
    """
    Run the pipeline for each gene and store results in CSV.
    """
    all_results = []


    print(f"Processing gene: {gene_name} ({len(gene_df)} samples)")

    df_gene = load_embeddings(gene_df,column_name="pooled_embedding")
    X, y = prepare_data(df_gene, embedding_column="pooled_embedding", label_column="Phenotype")
    print("Any NaNs in X?", np.isnan(X).any())
    print("Total NaNs in X:", np.isnan(X).sum())

    X_train, X_test, y_train, y_test = split_data(X, y)
    print(y_test)

#     # PCA and save visualization
#     visualize_pca(X_train, y_train, gene_name)

    # Hyperparameter tuning
    _, grid_list = run_grid_search(X_train, y_train)
    results = evaluate_models(grid_list, X_test, y_test, gene_name)

    best_grid = grid_list[2]  

    best_model = best_grid.best_estimator_
    joblib.dump(best_model, f"{gene_name}_best_rf_model.joblib")

    print(f"Best model saved as '{gene_name}_best_rf_model.joblib'")
    # Evaluate and collect results
    
    all_results.extend(results)

    # Save all results to CSV
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(os.path.join(RESULTS_DIR, f"{gene_name}_token_model_performance.csv"), index=False)

    print(f"\n All results saved in '{RESULTS_DIR}/model_performance.csv'")



## load mean embedding chunks and generate X and y

In [7]:
mean_npzs  = sorted(glob.glob(f"data/embeddings/{gene_name}/mean/{gene_name}_mean_chunk_*.npz"))
pheno_df   = pd.read_csv(
                f"./data/sequence_data_csv/{gene_name}_combined_sequence_data.csv",
                usecols=["Filename", "Phenotype"]
             ).set_index("Filename")

X_list, y_list = [], []

for f in mean_npzs:
    z      = np.load(f)
    ids    = z["identifier"]
    X_list.append(z["mean_embedding"])            # (n_i, 320)
    phenos = pheno_df.loc[ids, "Phenotype"].values
    y_list.append((phenos == "R").astype(np.int8))

X_all = np.concatenate(X_list)
y_all = np.concatenate(y_list)

np.save(f"training_data/{gene_name}_X_all.npy", X_all.astype("float32"))
np.save(f"training_data/{gene_name}_y_all.npy", y_all)
print("X_all / y_all written:", X_all.shape, y_all.shape)

## in the case it is saved already-use the following code
# Load feature matrix (X) and label array (y)
# X_all = np.load(f"training_data/{gene_name}_X_all.npy")
# y_all = np.load(f"training_data/{gene_name}_y_all.npy")

# print("Loaded shapes:")
# print("X_all:", X_all.shape)
# print("y_all:", y_all.shape)




X_all / y_all written: (3575, 320) (3575,)


In [8]:
# run model on the mean embedding
X_train, X_test, y_train, y_test = split_data(X_all, y_all)
# _, grid_list = run_grid_search(X_train, y_train)
_, search_list = run_random_search(X_train, y_train, n_iter=25, cv_folds=3)

results = evaluate_models(search_list, X_test, y_test, gene_name)

Running Randomized Search for KNeighborsRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Running Randomized Search for SVR …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Running Randomized Search for RandomForestRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Best Model for tlyA: KNeighborsRegressor(algorithm='brute', leaf_size=31, n_neighbors=29, p=1)
AUC: 0.5143

 -------------------------------------------------------------------------------- 

Best Model for tlyA: SVR(C=0.1)
AUC: 0.5153

 -------------------------------------------------------------------------------- 

Best Model for tlyA: RandomForestRegressor(max_features='log2', min_samples_leaf=5,
                      min_samples_split=10, n_estimators=98)
AUC: 0.5142

 -------------------------------------------------------------------------------- 



In [9]:
#save the trained models
# Create output directory if it doesn't exist
os.makedirs("trained_models", exist_ok=True)

# Save all models from the search_list
model_names = ["knn", "svr", "rf"]  # assumed order
for model_name, grid in zip(model_names, search_list):
    best_model = grid.best_estimator_
    model_path = f"trained_models/{gene_name}_best_{model_name}_model.joblib"
    joblib.dump(best_model, model_path)
    print(f"Saved: {model_path}")

Saved: trained_models/tlyA_best_knn_model.joblib
Saved: trained_models/tlyA_best_svr_model.joblib
Saved: trained_models/tlyA_best_rf_model.joblib


In [10]:
all_results=[]
all_results.extend(results)

# Save all results to CSV
results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(RESULTS_DIR, f"{gene_name}_token_model_performance.csv"), index=False)

## significance test

In [11]:
from sklearn.model_selection import StratifiedKFold
import joblib

# Cross-validation setup
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize result container
all_auc_records = []

model_names = ["knn", "svr", "rf"]  # maintain order
model_folds = {name: [] for name in model_names}

fold_id = 0

for train_idx, test_idx in kf.split(X_all, y_all):
    print(f"\n--- Fold {fold_id + 1} ---")

    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]

    # Run search
    _, search_list = run_random_search(X_train, y_train, n_iter=25, cv_folds=3)

    # Evaluate and save
    for model_name, grid in zip(model_names, search_list):
        best_model = grid.best_estimator_
        model_folds[model_name].append(best_model)

        try:
            preds = best_model.predict(X_test)
            auc = roc_auc_score(y_test, preds)
        except ValueError as e:
            auc = None
            print(f"[Warning] AUC error for {model_name}: {e}")

        all_auc_records.append({
            "Gene": gene_name,
            "Model": model_name,
            "Fold": fold_id + 1,
            "AUC": auc
        })

        print(f"Model: {model_name} | Fold {fold_id+1} AUC: {auc}")

        # Optional: save model
        model_out = f"trained_models/{gene_name}_fold{fold_id+1}_{model_name}.joblib"
        joblib.dump(best_model, model_out)

    fold_id += 1

# Save fold-wise AUCs
results_df = pd.DataFrame(all_auc_records)
results_df.to_csv(os.path.join(RESULTS_DIR, f"{gene_name}_cv_auc_per_fold.csv"), index=False)

# Print summary
summary = results_df.groupby("Model")["AUC"].agg(["mean", "std"]).reset_index()
print("\nCross-validation summary (mean ± std AUC):")
print(summary)



--- Fold 1 ---
Running Randomized Search for KNeighborsRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Running Randomized Search for SVR …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Running Randomized Search for RandomForestRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model: knn | Fold 1 AUC: 0.48127019386106623
Model: svr | Fold 1 AUC: 0.4956246634356489
Model: rf | Fold 1 AUC: 0.5187466343564888

--- Fold 2 ---
Running Randomized Search for KNeighborsRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits


/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Running Randomized Search for SVR …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Running Randomized Search for RandomForestRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model: knn | Fold 2 AUC: 0.5032028825943349
Model: svr | Fold 2 AUC: 0.5030694291529043
Model: rf | Fold 2 AUC: 0.5013762386147532

--- Fold 3 ---
Running Randomized Search for KNeighborsRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits


/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Running Randomized Search for SVR …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Running Randomized Search for RandomForestRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model: knn | Fold 3 AUC: 0.49406132185633733
Model: svr | Fold 3 AUC: 0.5114269509224969
Model: rf | Fold 3 AUC: 0.511410269242318

--- Fold 4 ---
Running Randomized Search for KNeighborsRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits


/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Running Randomized Search for SVR …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Running Randomized Search for RandomForestRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model: knn | Fold 4 AUC: 0.5051546391752577
Model: svr | Fold 4 AUC: 0.4971474326894204
Model: rf | Fold 4 AUC: 0.4971474326894204

--- Fold 5 ---
Running Randomized Search for KNeighborsRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits


/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Running Randomized Search for SVR …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Running Randomized Search for RandomForestRegressor …
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Model: knn | Fold 5 AUC: 0.5205351483001369
Model: svr | Fold 5 AUC: 0.5189503886831482
Model: rf | Fold 5 AUC: 0.5173656290661596

Cross-validation summary (mean ± std AUC):
  Model      mean       std
0   knn  0.500845  0.014500
1    rf  0.509209  0.009607
2   svr  0.505244  0.009860
